<div align="center">

# Preprocessing

</div>

In [1]:
import h5py
import numpy as np
import mne
import os

def filter_hdf5_file(h5_path, out_path, highpass=0.5, notch=60):

    with h5py.File(h5_path, "r") as f:
        signals = f["signals"][:]          # (ch, samples)
        fs = f["sampling_rate"][()]
        ch_names = [c.decode() for c in f["channel_names"][:]]
        seizures = f["seizure_intervals"][:]

    # -----------------------------
    # Create MNE object
    # -----------------------------
    info = mne.create_info(
        ch_names=ch_names,
        sfreq=fs,
        ch_types="eeg"
    )

    raw = mne.io.RawArray(signals, info, verbose=False)

    # -----------------------------
    # High-pass filter (0.5 Hz)
    # -----------------------------
    raw.filter(
        l_freq=highpass,
        h_freq=None,
        method="iir",
        iir_params=dict(order=4, ftype="butter"),
        phase="zero",
        verbose=False
    )

    # -----------------------------
    # Notch filter (60 Hz)
    # -----------------------------
    raw.notch_filter(
        freqs=notch,
        method="iir",
        phase="zero",
        verbose=False
    )

    filtered = raw.get_data()

    # -----------------------------
    # Save filtered HDF5
    # -----------------------------
    os.makedirs(os.path.dirname(out_path), exist_ok=True)

    with h5py.File(out_path, "w") as f:
        f.create_dataset("signals", data=filtered, compression="gzip")
        f.create_dataset("sampling_rate", data=fs)
        f.create_dataset(
            "channel_names",
            data=np.array(ch_names, dtype="S")
        )
        f.create_dataset("seizure_intervals", data=seizures)

    print(f"Saved filtered → {out_path}")

In [ ]:
import glob
import os

input_dir = "../Dataset_hdf5_clean"
output_dir = "../Dataset_hdf5_filtered"

# Create output directory if not exists
os.makedirs(output_dir, exist_ok=True)

files = glob.glob(input_dir + "/**/*.h5", recursive=True)

if not files:
    print(f"No .h5 files found in {input_dir}")
else:
    print(f"Found {len(files)} files. Processing...")
    for f in files:
        out = f.replace("Dataset_hdf5", "Dataset_hdf5_filtered")
        
        # Create output subdirectory
        os.makedirs(os.path.dirname(out), exist_ok=True)
        
        try:
            filter_hdf5_file(f, out)
            print(f"✓ Filtered: {os.path.basename(f)}")
        except Exception as e:
            print(f"✗ Error filtering {f}: {e}")

Found 655 files. Processing...
Saved filtered → ../Dataset_hdf5_filtered_clean\chb01\chb01_01.h5
✓ Filtered: chb01_01.h5
Saved filtered → ../Dataset_hdf5_filtered_clean\chb01\chb01_02.h5
✓ Filtered: chb01_02.h5
Saved filtered → ../Dataset_hdf5_filtered_clean\chb01\chb01_03.h5
✓ Filtered: chb01_03.h5
Saved filtered → ../Dataset_hdf5_filtered_clean\chb01\chb01_04.h5
✓ Filtered: chb01_04.h5
Saved filtered → ../Dataset_hdf5_filtered_clean\chb01\chb01_05.h5
✓ Filtered: chb01_05.h5
Saved filtered → ../Dataset_hdf5_filtered_clean\chb01\chb01_06.h5
✓ Filtered: chb01_06.h5
Saved filtered → ../Dataset_hdf5_filtered_clean\chb01\chb01_07.h5
✓ Filtered: chb01_07.h5
Saved filtered → ../Dataset_hdf5_filtered_clean\chb01\chb01_08.h5
✓ Filtered: chb01_08.h5
Saved filtered → ../Dataset_hdf5_filtered_clean\chb01\chb01_09.h5
✓ Filtered: chb01_09.h5
Saved filtered → ../Dataset_hdf5_filtered_clean\chb01\chb01_10.h5
✓ Filtered: chb01_10.h5
Saved filtered → ../Dataset_hdf5_filtered_clean\chb01\chb01_11.h5
✓ F